In [1]:
%load_ext sql
%sql sqlite:///advanced_sql.db

Connecting to 'sqlite:///advanced_sql.db'

In [2]:
%%sql
CREATE TABLE vehicle_telemetry (
    vehicle_id INT,
    speed INT,
    event_timestamp DATETIME
);

INSERT INTO vehicle_telemetry (vehicle_id, speed, event_timestamp) VALUES
(1, 45, '2026-03-13 10:00:00'),
(1, 55, '2026-03-13 10:05:00'),
(1, 50, '2026-03-13 10:10:00'), -- This is the latest for vehicle 1
(2, 30, '2026-03-13 09:00:00'),
(2, 0, '2026-03-13 09:30:00');  -- This is the latest for vehicle 2

Running query in 'sqlite:///advanced_sql.db'

5 rows affected.

++
||
++
++

In [3]:
%%sql
WITH RankedTelemetry AS (
    SELECT 
        vehicle_id,
        speed,
        event_timestamp,
        ROW_NUMBER() OVER(PARTITION BY vehicle_id ORDER BY event_timestamp DESC) as rn
    FROM vehicle_telemetry
)
SELECT 
    vehicle_id, 
    speed, 
    event_timestamp
FROM RankedTelemetry
WHERE rn = 1;

Running query in 'sqlite:///advanced_sql.db'

vehicle_id,speed,event_timestamp
1,50,2026-03-13 10:10:00
2,0,2026-03-13 09:30:00


In [4]:
%%sql
CREATE TABLE inference_logs (
    model_id VARCHAR(50),
    status VARCHAR(10),
    log_timestamp DATETIME
);

INSERT INTO inference_logs (model_id, status, log_timestamp) VALUES
('pedestrian_v1', 'SUCCESS', '2026-03-13 10:00:00'),
('pedestrian_v1', 'FAIL', '2026-03-13 10:05:00'),
('pedestrian_v1', 'FAIL', '2026-03-13 10:10:00'), -- pedestrian_v1 has a 66% failure rate
('lane_assist_v2', 'SUCCESS', '2026-03-13 10:00:00'),
('lane_assist_v2', 'SUCCESS', '2026-03-13 10:05:00'),
('lane_assist_v2', 'SUCCESS', '2026-03-13 10:10:00'), -- lane_assist_v2 has a 0% failure rate
('pedestrian_v1', 'SUCCESS', '2026-03-12 10:00:00'); -- Older date to test the WHERE clause

Running query in 'sqlite:///advanced_sql.db'

7 rows affected.

++
||
++
++

In [5]:
%%sql
SELECT * FROM inference_logs;

Running query in 'sqlite:///advanced_sql.db'

model_id,status,log_timestamp
pedestrian_v1,SUCCESS,2026-03-13 10:00:00
pedestrian_v1,FAIL,2026-03-13 10:05:00
pedestrian_v1,FAIL,2026-03-13 10:10:00
lane_assist_v2,SUCCESS,2026-03-13 10:00:00
lane_assist_v2,SUCCESS,2026-03-13 10:05:00
lane_assist_v2,SUCCESS,2026-03-13 10:10:00
pedestrian_v1,SUCCESS,2026-03-12 10:00:00


In [14]:
%%sql
select model_id, sum(case when status = 'FAIL' then 1 else 0 end) *100.0  / count(*) as failure_rate
from inference_logs
WHERE date(log_timestamp) = '2026-03-13'
group by model_id
HAVING sum(case when status = 'FAIL' THEN 1 else 0 end) *100.0 / count(*) > 5.0;

Running query in 'sqlite:///advanced_sql.db'

model_id,failure_rate
pedestrian_v1,66.66666666666667


In [9]:
%%sql
SELECT 
    model_id,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as failure_rate
FROM inference_logs
WHERE DATE(log_timestamp) = '2026-03-13'
GROUP BY model_id
HAVING SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) > 5.0;

Running query in 'sqlite:///advanced_sql.db'

model_id,failure_rate
pedestrian_v1,66.66666666666667
